# Обучение голоса в Google Colab

Бесплатный Colab: T4 (16 ГБ), сессия до 12 часов, отключение примерно после
90 минут простоя, фонового выполнения нет — вкладку нельзя закрывать, GPU
не гарантирован.

Поэтому всё состояние обучения пишется на Google Drive, а после обрыва сессии
обучение продолжается ячейкой «Продолжить обучение» с последнего чекпоинта.

Порядок: GPU → Drive → установка → запись → профиль → обучение → проверка → скачивание.


## 1. Проверить, что GPU выдали


In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), (
    'GPU не выдан: Runtime -> Change runtime type -> T4 GPU, потом перезапустите ячейку'
)
print('GPU:', torch.cuda.get_device_name(0))


## 2. Подключить Drive

Всё, что нужно сохранить между сессиями, живёт здесь.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, pathlib
WORK = pathlib.Path('/content/drive/MyDrive/voice-clone-tts')
for name in ('input', 'profiles', 'voices', 'logs'):
    (WORK / name).mkdir(parents=True, exist_ok=True)
print('рабочая папка:', WORK)


## 3. Установка

`VC = 'rvc'` — Applio (актуальный RVC, ближе к оригиналу, установка дольше).  
`VC = 'sovits'` — so-vits-svc-fork (одна команда, апстрим не развивается).


In [ ]:
VC = 'rvc'   # 'rvc' или 'sovits'

!git clone --depth 1 -b claude/voice-cloning-text-synthesis-7mnlwx https://github.com/tyetladd/RustTraining /content/RustTraining 2>/dev/null || true
%pip install -q -e '/content/RustTraining/voice-clone-tts[stress,asr,silero]'

if VC == 'rvc':
    !git clone --depth 1 https://github.com/IAHispano/Applio /content/Applio 2>/dev/null || true
    %pip install -q -r /content/Applio/requirements.txt
    # Сборки torch 2.11 с PyPI собраны под CUDA 13, а в образе лежит CUDA 12.8:
    # torchaudio не находит libcudart.so.13 и не импортируется, а без него
    # обучение не стартует. Переставляем пару из индекса PyTorch под cu128.
    %pip install -q -U --index-url https://download.pytorch.org/whl/cu128 torch==2.11.0 torchaudio==2.11.0
    os.environ['VCTTS_APPLIO_DIR'] = '/content/Applio'

    # git clone не тянет веса: предикторы f0, эмбеддер и предобученные
    # модели скачивает отдельная команда. Без них питч-экстракция молча
    # не пишет ни одного файла, а обучение остаётся без данных.
    !cd /content/Applio && python3 core.py prerequisites
else:
    %pip install -q -U so-vits-svc-fork

!vctts converters


### Проверка окружения после установки

Ворох сообщений вида «X requires Y, but you have Z» выше — это pip, ругающийся на
*другие* пакеты образа (bigframes, ydata-profiling, google-colab и прочие), которыми
мы не пользуемся. Значение имеет одно: импортируются ли `torch` и `torchaudio` в
свежем процессе — именно так их увидит обучение, которое идёт отдельными процессами.

Ячейка ниже проверяет это и говорит, что делать, если нет.


In [ ]:
import json, subprocess, sys

PROBE = """
import json
out = {}
try:
    import torch
    out['torch'] = torch.__version__
    out['cuda'] = torch.cuda.is_available()
except Exception as exc:
    out['torch'] = f'ОШИБКА: {exc.__class__.__name__}: {exc}'
for name in ('torchaudio', 'torchvision'):
    try:
        out[name] = __import__(name).__version__
    except Exception as exc:
        out[name] = f'ОШИБКА: {exc.__class__.__name__}: {exc}'
print(json.dumps(out))
"""

# Важно: опрашиваем НОВЫЙ процесс. В текущем ядре torch уже импортирован, и оно
# покажет старую версию, даже если установка её заменила. Обучение тоже идёт в
# отдельных процессах — значит, видеть мы должны именно эту картину.
probe = subprocess.run([sys.executable, '-c', PROBE], capture_output=True, text=True)
fresh = json.loads(probe.stdout) if probe.stdout.strip().startswith('{') else {}
if not fresh:
    print('не удалось опросить окружение:\n', probe.stderr[-2000:])
else:
    for key, value in fresh.items():
        print(f'  {key:12}: {value}')

problems = [k for k, v in fresh.items() if str(v).startswith('ОШИБКА')]
if 'torchaudio' in problems:
    print('\n!! torchaudio не импортируется — Applio использует его для FCPE,')
    print('   обучение не стартует. Обычно это несовпадение сборки CUDA:')
    print('   %pip install -q -U --index-url https://download.pytorch.org/whl/cu128 \\')
    print('        torch==2.11.0 torchaudio==2.11.0')
elif 'torchvision' in problems:
    print('\n!! torchvision не совместим с установленным torch. Нашему пути он не нужен:')
    print('   %pip uninstall -y torchvision')
elif fresh.get('cuda') is False:
    print('\n!! CUDA не видна из свежего процесса — проверьте, что GPU включён')
else:
    print('\nокружение в порядке, можно продолжать')


Предупреждения pip о конфликте версий здесь ожидаемы: Applio жёстко пинит torch
и transformers поверх предустановленных в Colab. Если после установки импорт torch
падает — Runtime → Restart session и выполните ячейки заново начиная со 2-й
(клонирование и установка повторно не нужны, кроме `os.environ[...]`).


## 4. Запись голоса

Нужно 5–30 минут чистой речи одного человека без музыки и шума. Положите файл
в `MyDrive/voice-clone-tts/input/` через веб-интерфейс Drive — для больших файлов
это надёжнее, чем загрузка из браузера.


In [ ]:
# Загрузка прямо из браузера (для файлов до ~100 МБ):
# from google.colab import files
# for name, blob in files.upload().items():
#     (WORK / 'input' / name).write_bytes(blob)

SOURCE = sorted((WORK / 'input').glob('*.*'))[0]
SPEAKER = None   # имя голоса; None — взять из имени файла записи
print('запись:', SOURCE)

import re

# Имя голоса: заданное в параметрах либо из имени файла записи — так метка
# модели не остаётся дефолтной из примера.
SPEAKER = SPEAKER or re.sub(r'[^a-z0-9]+', '_', SOURCE.stem.lower()).strip('_') or 'voice'
print('голос :', SPEAKER)


### Что показывает запись

Вердикт по длительности, формату, уровню, клиппингу и SNR — до начала обучения.
Прогон не останавливается даже при `✗`: решать вам. Но если здесь написано
«лучше переписать», то часы GPU уйдут на запись, которую всё равно придётся
переписывать.


In [ ]:
cmd = f'vctts check "{SOURCE}" --purpose vc'
!{cmd}


## 5. Профиль диктора

Отчёт о качестве записи и определение языка. Если предупреждает про SNR или
клиппинг — замените запись сейчас, а не после часов обучения.


In [ ]:
cmd = f'vctts profile build "{SOURCE}" -o "{WORK}/profiles/{SPEAKER}" --name {SPEAKER} --overwrite'
!{cmd}


## 6. Обучение

300 эпох на ~10 минутах речи — это несколько часов на T4. `save_every_epoch`
определяет, сколько работы потеряется при обрыве; `batch_size` при нехватке
памяти уменьшайте до 4.


In [ ]:
EPOCHS = 300
BATCH_SIZE = 8
SAVE_EVERY = 25

def train_command(resume=False, epochs=None):
    """Собрать команду обучения: у драйверов разные полезные опции."""
    parts = [
        'vctts voice train',
        f'-p "{WORK}/profiles/{SPEAKER}"',
        f'-o "{WORK}/voices/{SPEAKER}"',
        f'--vc {VC}',
        f'--epochs {epochs or EPOCHS}',
        '--resume' if resume else '--overwrite',
        f'--vc-option batch_size={BATCH_SIZE}',
    ]
    if VC == 'rvc':
        # Чекпоинты Applio должны лежать вне эфемерного чекаута.
        parts += [f'--vc-option logs_dir="{WORK}/logs"',
                  f'--vc-option save_every_epoch={SAVE_EVERY}']
    else:
        # so-vits-svc хранит всё внутри -o, эта папка уже персистентная.
        parts += ['--sample-rate 44100']
    return ' '.join(parts)

print(train_command())


In [ ]:
cmd = train_command()
!{cmd}


### Продолжить обучение после обрыва

В новой сессии выполните ячейки 1–5 (установка нужна заново, Drive — нет), затем
эту: подготовка датасета и извлечение признаков пропускаются, обучение идёт
с последнего чекпоинта на Drive.


In [ ]:
cmd = train_command(resume=True)
!{cmd}


## 7. Проверка: Silero TTS + ваш голос


In [ ]:
TEXT = 'Проверка синтеза. Старинный замок на горе, а на двери замок.'

cmd = (f'vctts speak -b silero -l ru --voice-model "{WORK}/voices/{SPEAKER}" '
       f'--transpose auto -t "{TEXT}" -o "{WORK}/sample.wav"')
!{cmd}

from IPython.display import Audio
Audio(str(WORK / 'sample.wav'))


Если тембр «плывёт», попробуйте фиксированный сдвиг вместо `auto`
(`--transpose -10` для мужского голоса поверх женского диктора Silero) или
смените исходного диктора: `--backend-option voice=aidar` — мужской.
Чем ближе диктор Silero к целевому по высоте, тем чище работает конверсия.


## 8. Забрать модель

Модель уже на Drive; архив нужен, чтобы перенести её на свою машину. Локально
для синтеза понадобится тот же тулчейн (Applio или so-vits-svc-fork).


In [ ]:
import shutil
archive = shutil.make_archive(f'/content/{SPEAKER}-voice', 'zip', WORK / 'voices' / SPEAKER)
print(archive)
from google.colab import files
files.download(archive)
